# 01 — Data Exploration
## Load, Validate, and Understand Your DataGolf CSVs

**Purpose:** Verify your data pipeline works, understand what you have, and catch issues early.

**What this notebook does:**
1. Loads all CSV files via `DataLoader` with schema validation
2. Summarizes data shape, coverage, and quality
3. Explores SG distributions and player/tournament counts
4. Identifies missing data and potential issues

**Prerequisites:**
- CSV files placed in your `DATA_DIR` (configured in `config/settings.py`)
- Conda environment activated: `conda activate golf_model`


In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import Settings
from data.loader import DataLoader
from data.schemas import RoundsSchema, EventsSchema, OddsSchema, validate_dataframe

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

cfg = Settings()
print(f"Project root: {cfg.PROJECT_ROOT}")
print(f"Data dir:     {cfg.DATA_DIR}")
print(f"Train:        {cfg.train_season_range}")
print(f"Holdout:      {cfg.holdout_season_range}")

# Validate config
warnings = cfg.validate()
if warnings:
    for w in warnings:
        print(f"⚠ {w}")
else:
    print("✓ Configuration valid")


In [ ]:
# --- Discover available files ---
loader = DataLoader(cfg)
print("Available CSV files:")
for f in loader.list_available_files():
    print(f"  • {f}")


In [ ]:
# --- Load rounds (core dataset) ---
rounds_df = loader.load_rounds()
print(f"\nRounds shape: {rounds_df.shape}")
print(f"Columns: {list(rounds_df.columns)}")
rounds_df.head()


In [ ]:
# --- Schema validation ---
issues = validate_dataframe(rounds_df, RoundsSchema())
if issues:
    print("Schema issues found:")
    for issue in issues:
        print(f"  ⚠ {issue}")
else:
    print("✓ Rounds schema validation passed")

# Rename DataGolf native columns to legacy names used in model code
rounds_df = rounds_df.rename(columns={
    "dg_id": "player_id",
    "event_completed": "date",
    "year": "calendar_year",
    "round_score": "gross_score",
    "fin_text": "finish_position_raw",
})

# Derive made_cut from finish position
if "finish_position_raw" in rounds_df.columns:
    rounds_df["made_cut"] = rounds_df["finish_position_raw"].str.upper() != "CUT"

print(f"\nRenamed columns. Shape: {rounds_df.shape}")
print(f"Columns: {list(rounds_df.columns)}")


In [ ]:
# --- Data coverage summary ---
print("=== ROUNDS COVERAGE ===")
if "season" in rounds_df.columns:
    print(f"Seasons:     {sorted(rounds_df['season'].dropna().unique().astype(int))}")
print(f"Players:     {rounds_df['player_id'].nunique():,}")
print(f"Events:      {rounds_df['event_id'].nunique():,}")
print(f"Total rows:  {len(rounds_df):,}")

# Date is event_completed (tournament end date), not per-round date
if "date" in rounds_df.columns:
    dates = pd.to_datetime(rounds_df["date"], errors="coerce")
    valid_dates = dates.dropna()
    if len(valid_dates) > 0:
        print(f"Date range:  {valid_dates.min().date()} to {valid_dates.max().date()}")

# Tour breakdown
if "tour" in rounds_df.columns:
    print(f"\nTours: {rounds_df['tour'].value_counts().to_dict()}")

# Missing data report
print("\n=== MISSING DATA ===")
for col in rounds_df.columns:
    n_miss = rounds_df[col].isna().sum()
    n_empty = (rounds_df[col].astype(str).str.strip() == "").sum()
    total = n_miss + n_empty
    if total > 0:
        pct = total / len(rounds_df) * 100
        print(f"  {col}: {total:,} missing/empty ({pct:.1f}%)")


In [ ]:
# --- SG distributions ---
sg_cols = [c for c in ["sg_total", "sg_ott", "sg_app", "sg_arg", "sg_putt"] if c in rounds_df.columns]

if sg_cols:
    fig, axes = plt.subplots(1, len(sg_cols), figsize=(4 * len(sg_cols), 4))
    if len(sg_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, sg_cols):
        data = rounds_df[col].dropna()
        ax.hist(data, bins=60, alpha=0.7, color="#2563eb", edgecolor="white")
        ax.axvline(x=0, color="red", linestyle="--", alpha=0.5)
        ax.set_title(f"{col}\nμ={data.mean():.3f}, σ={data.std():.3f}")
        ax.set_xlabel("Strokes Gained")

    plt.suptitle("Strokes Gained Distributions", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary stats
    print("\n=== SG SUMMARY STATISTICS ===")
    print(rounds_df[sg_cols].describe().round(3).to_string())
else:
    print("No SG columns found — check your CSV column names against RoundsSchema")


In [ ]:
# --- Rounds per player distribution ---
rounds_per_player = rounds_df.groupby("player_id").size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(rounds_per_player, bins=50, alpha=0.7, color="#2563eb", edgecolor="white")
ax.axvline(x=cfg.MIN_ROUNDS_FOR_ESTIMATE, color="red", linestyle="--",
           label=f"Min for estimate ({cfg.MIN_ROUNDS_FOR_ESTIMATE})")
ax.set_xlabel("Number of rounds")
ax.set_ylabel("Number of players")
ax.set_title("Rounds per Player")
ax.legend()
plt.tight_layout()
plt.show()

n_above = (rounds_per_player >= cfg.MIN_ROUNDS_FOR_ESTIMATE).sum()
print(f"Players with ≥{cfg.MIN_ROUNDS_FOR_ESTIMATE} rounds: {n_above} / {len(rounds_per_player)}")


In [ ]:
# --- Load events (if available) ---
try:
    events_df = loader.load_events()
    print(f"Events shape: {events_df.shape}")
    events_df.head()
except FileNotFoundError as e:
    print(f"Events data not found: {e}")
    events_df = None


In [ ]:
# --- Load odds (if available) ---
try:
    odds_df = loader.load_odds()
    print(f"Odds shape: {odds_df.shape}")
    print(f"Columns: {list(odds_df.columns)}")
    
    # Check for bookmaker column (native DG name)
    book_col = "bookmaker" if "bookmaker" in odds_df.columns else "book"
    if book_col in odds_df.columns:
        print(f"\nSportsbooks: {odds_df[book_col].unique().tolist()}")
    
    # Check for odds column (native DG name)
    odds_col = "close_odds" if "close_odds" in odds_df.columns else "decimal_odds"
    if odds_col in odds_df.columns:
        print(f"\nOdds range: {odds_df[odds_col].astype(float).min():.1f} to {odds_df[odds_col].astype(float).max():.1f}")
    
    if "market" in odds_df.columns:
        print(f"\nMarkets: {odds_df['market'].unique().tolist()}")
    
    odds_df.head()
except FileNotFoundError as e:
    print(f"Odds data not found: {e}")
    odds_df = None


---
## ✅ Data Exploration Complete

**Next steps:**
- If schema issues were found, fix your CSVs or update `data/schemas.py`
- If column names don't match, update the mapping in `data/loader.py`
- Once data loads cleanly → proceed to **02_feature_engineering.ipynb**
